In [ ]:
from pathlib import Path
from pprint import pprint

from openff.pablo import (
    CCD_RESIDUE_DEFINITION_CACHE,
    ResidueDefinition,
    topology_from_pdb,
)
from openff.pablo.chem import PEPTIDE_BOND
from openff.toolkit import Molecule

In [ ]:
# Construct a resdef of the dye with atom names that differ from the PDB file
smiles = "[c:1]1([H:41])[c:2]([H:42])[c:3]2[c:4]([c:5]([H:43])[c:6]1[N:7]1[C:8](=[O:9])[C:10]([H:44])([H:45])[C@@:11]([S:12][C:13]([C@:14]([N:15]([H:16])[H:50])([C:17](=[O:18])[O:19][H:59])[H:49])([H:47])[H:48])([H:46])[C:20]1=[O:21])[C:22](=[O:23])[O:24][C:25]21[c:26]2[c:27]([c:28]([H:51])[c:29]([O:32][H:54])[c:30]([H:52])[c:31]2[H:53])[O:33][c:34]2[c:35]1[c:36]([H:55])[c:37]([H:56])[c:38]([O:40][H:58])[c:39]2[H:57]"

substructure_mol = Molecule.from_mapped_smiles(smiles, allow_undefined_stereo=True)
substructure_mol.generate_unique_atom_names()
linking_bond = PEPTIDE_BOND
for i, atom in enumerate(substructure_mol.atoms):
    if i == 14:
        atom.name = linking_bond.atom2
        # linking_bond = linking_bond.replace(atom2=atom.name)
    if i == 16:
        atom.name = linking_bond.atom1
        # linking_bond = linking_bond.replace(atom1=atom.name)
    if i in {15, 18, 58}:
        atom.metadata["substructure_atom"] = False
        atom.metadata["leaving_atom"] = True
    else:
        atom.metadata["substructure_atom"] = True
        atom.metadata["leaving_atom"] = False
assert "CA" not in {atom.name for atom in substructure_mol.atoms}
assert "H" not in {atom.name for atom in substructure_mol.atoms}

dye_res_def = ResidueDefinition.from_molecule(
    molecule=substructure_mol,
    residue_name="DYE",
    linking_bond=linking_bond,
    description="CYSTEINE-CONJUGATED FLUOROPHORE MALEIMIDE",
)

In [ ]:
path = Path("../openff/pablo/_tests/data/3ip9_dye_trimmed.pdb")
my_resdb = CCD_RESIDUE_DEFINITION_CACHE.with_([dye_res_def])

In [ ]:
# Try to load it
assert "DYE" not in CCD_RESIDUE_DEFINITION_CACHE
pablo_top = topology_from_pdb(
    path,
    residue_database=my_resdb,
)
assert "DYE" in [res.identifier[3] for res in pablo_top.molecule(0).residues]

## Dig deeper

In [ ]:
from openff.pablo._pdb_data import PdbData

data = PdbData.from_file(path)

In [ ]:
matches = list(data.match_residues(my_resdb, [], []))
pprint(matches)

In [ ]:
from openff.pablo._matching import only_matched
from openff.pablo._utils import sort_tuple

dye_matches = list(only_matched(matches[1]))

prototype_match, other_matches = dye_matches[0], dye_matches[1:]
ptype_canonical_name_to_idx = {
    atom.name: i for i, atom in prototype_match.index_to_atomdef.items()
}
ptype_bonds = {
    sort_tuple(
        (
            ptype_canonical_name_to_idx[bond.atom1],
            ptype_canonical_name_to_idx[bond.atom2],
        )
    ): bond
    for bond in prototype_match.residue_definition.bonds
    if bond.atom1 in ptype_canonical_name_to_idx
    and bond.atom2 in ptype_canonical_name_to_idx
}

for i, match in enumerate(dye_matches):
    print(match.description, i)
    if i == 0:
        print("  PROTOTYPE")
    print(f"  {match.agrees_with(prototype_match)=}")
    print(f"  {match.expects_crosslink=}")
    print(f"  {match.expects_posterior_bond=}")
    print(f"  {match.expects_prior_bond=}")
    print(
        f"  Number of matched atom names: {sum(data.name[pdb_idx] == match.index_to_atomdef[pdb_idx].name for pdb_idx in match.res_atom_idcs)}"
    )
    for pdb_idx, atom in match.index_to_atomdef.items():
        prototype_atom = prototype_match.index_to_atomdef[pdb_idx]
        if i == 0:
            print(
                f"{pdb_idx: >7} ({data.name[pdb_idx]:>4}): {prototype_atom.name: <4} {atom.symbol: >4}{atom.charge:+} "
            )
        if prototype_atom != atom:
            print(
                f"{pdb_idx: >7} ({data.name[pdb_idx]:>4}): {prototype_atom.name: <4} -> {atom.name: <4}"
            )
        if prototype_atom.charge != atom.charge:
            print(f"         {prototype_atom.charge: <2} -> {atom.charge: <2}")
        if prototype_atom.symbol != atom.symbol:
            print(f"         {prototype_atom.symbol: <2} -> {atom.symbol: <2}")

    canonical_name_to_idx = {atom.name: i for i, atom in match.index_to_atomdef.items()}
    for bond in match.residue_definition.bonds:
        if (
            bond.atom1 not in canonical_name_to_idx
            or bond.atom2 not in canonical_name_to_idx
        ):
            continue
        atom1_idx = canonical_name_to_idx[bond.atom1]
        atom2_idx = canonical_name_to_idx[bond.atom2]

        ptype_bond = ptype_bonds[sort_tuple((atom1_idx, atom2_idx))]
        if i == 0:
            print(f"     {sort_tuple((atom1_idx, atom2_idx))}: {bond.order}")

        if ptype_bond.order != bond.order:
            print(
                f"     {sort_tuple((atom1_idx, atom2_idx))}: {ptype_bond.order} -> {bond.order}"
            )

In [ ]:
import numpy as np
from openff.units import unit

mols = []
for match in dye_matches:
    mol = match.residue_definition.to_openff_molecule()
    name_to_indices = {v.name: k for k, v in match.index_to_atomdef.items()}
    positions = [(0, 0, 0)] * mol.n_atoms
    for i, atom in enumerate(mol.atoms):
        if atom.name in name_to_indices:
            pdb_idx = name_to_indices[atom.name]
        elif atom.name == "H1x":
            pdb_idx = 3
        elif atom.name == "O3x":
            pdb_idx = 73
        elif atom.name == "H20x":
            pdb_idx = 74
        else:
            print(name)
            continue
        atom.metadata["pdb_index"] = pdb_idx
        positions[i] = (data.x[pdb_idx], data.y[pdb_idx], data.z[pdb_idx])
    mol.add_conformer(np.asarray(positions) * unit.angstrom)
    mols.append(mol)

In [ ]:
import numpy as np
from openff.units import unit

ref_mol, molecules = mols[0], mols[1:]
ref_mol = Molecule(ref_mol)
for mol in molecules:
    is_isomorphic, mapping = Molecule.are_isomorphic(mol, ref_mol, return_atom_map=True)
    assert is_isomorphic
    positions = [(0, 0, 0)] * ref_mol.n_atoms
    for i, j in mapping.items():
        positions[j] = mol.conformers[0][i].m_as(unit.angstrom)
    ref_mol.add_conformer(np.asarray(positions) * unit.angstrom)

w = ref_mol.visualize("nglview")
w.clear_representations()
w.add_licorice(multipleBond="symmetric", radius=0.3)
w.add_label(label_type="atomname", color="black")
w

In [ ]:
problematic_mol = Molecule.from_smiles("[NH2+]=CCCCCCC-[NH2]")
problematic_mol.generate_unique_atom_names()
problematic_mol.generate_conformers(n_conformers=1)
problematic_mol.to_topology().to_file(
    "../openff/pablo/_tests/data/conect_match_problematic_symmetry.pdb"
)

In [ ]:
problematic_mol_names = Molecule(problematic_mol)
for i, atom in enumerate(problematic_mol_names.atoms):
    atom.name = atom.symbol + str(i)

import logging
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

pablo_top = topology_from_pdb(
    "../openff/pablo/_tests/data/conect_match_problematic_symmetry.pdb",
    residue_database={"UNK": 
        [ResidueDefinition.from_molecule(
            residue_name="UNK", molecule=problematic_mol_names
        )]
    },
)

In [ ]:
pablo_top.molecule(0).visualize()